In [130]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import HTML
import xgboost as xgb
from category_encoders import MEstimateEncoder

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, FunctionTransformer
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer

plt.rc("figure", autolayout=True)
plt.rc(
    "axes",
    labelweight="bold",
    labelsize="large",
    titleweight="bold",
    titlesize=14,
    titlepad=10,
)

set_config(transform_output='pandas')

data = pd.read_csv('train.csv', index_col='Id')

In [131]:
outliers_index = data.loc[data['TotalBsmtSF'] > 3000].index.union(
    data.loc[data['SalePrice'] < 50000].index).union(
        data.loc[data['SalePrice'] > 700000].index)
data = data.drop(outliers_index)

X = data.drop('SalePrice', axis=1)
y = data.SalePrice

# data['TotalArea'] = data['TotalBsmtSF'] + data['GrLivArea']
# data['normTotalArea'] = (data['TotalArea'] - data['TotalArea'].mean())/data['TotalArea'].std()
# data['sqrtTotalArea'] = np.sqrt(data['normTotalArea'])
# sns.boxplot(data['sqrtTotalArea'])
# HTML(data.sort_values(by='TotalArea').tail(5).to_html())

In [132]:
m_estimate_cols = ['Neighborhood']
m_encoder = MEstimateEncoder(m=2.0)

X_encode_base = X[m_estimate_cols].sample(frac=0.15)
while (X_encode_base[m_estimate_cols].nunique() < X[m_estimate_cols].nunique()).all():
    X_encode_base = X[m_estimate_cols].sample(frac=0.15)
y_encode_base = y.loc[X_encode_base.index]
X = X.drop(X_encode_base.index)
y = y.drop(X_encode_base.index)

m_encoder.fit(X_encode_base, y_encode_base)
X[m_estimate_cols] = m_encoder.transform(X[m_estimate_cols])

In [133]:
RoofMatl_other = ['ClyTile', 'Membran', 'Metal', 'Roll', 'Tar&Grv']
RoofMatl_wood = ['WdShake', 'WdShngl']
RoofMatl_cat_map = dict(zip(RoofMatl_wood, ['Wood']*len(RoofMatl_wood))) | dict(zip(RoofMatl_other, ['Other']*len(RoofMatl_other)))

Foundation_other = ['Stone', 'Wood']
Foundation_cat_map = dict(zip(Foundation_other, ['Other']*len(Foundation_other)))

RoofStyle_other = ['Flat', 'Gambrel', 'Mansard', 'Shed']
RoofStyle_cat_map = dict(zip(RoofStyle_other, ['Other']*len(RoofStyle_other)))

BldgType_twnhs = ['TwnhsE']
BldgType_cat_map = dict(zip(BldgType_twnhs, ['Twnhs']*len(BldgType_twnhs)))

LotConfig_other = ['Corner', 'FR2', 'FR3']
LotConfig_cat_map = dict(zip(LotConfig_other, ['Other']*len(LotConfig_other)))

Condition1_other = ['RRNn', 'RRAn', 'PosN', 'PosA', 'RRNe', 'RRAe']
Condition1_cat_map = dict(zip(Condition1_other, ['Other']*len(Condition1_other)))

SaleType_WD = ['CWD', 'VWD']
SaleType_Oth = ['ConLw', 'ConLI', 'ConLD', 'Con']
SaleType_cat_map = dict(zip(SaleType_WD, ['WD']*len(SaleType_WD))) | dict(zip(SaleType_Oth, ['Oth']*len(SaleType_Oth)))

Exterior1st_Rare = ['BrkComm', 'Stone', 'AsphShn', 'ImStucc', 'CBlock']
Exterior1st_cat_map = dict(zip(Exterior1st_Rare, ['Rare']*len(Exterior1st_Rare)))

replace_cats_cols = ['RoofMatl', 'Foundation', 'RoofStyle', 'BldgType', 'LotConfig',
                     'Condition1', 'SaleType', 'Exterior1st']
cat_maps = [RoofMatl_cat_map, Foundation_cat_map, RoofStyle_cat_map, BldgType_cat_map, LotConfig_cat_map,
            Condition1_cat_map, SaleType_cat_map, Exterior1st_cat_map]
cat_map = dict(zip(replace_cats_cols, cat_maps))

def replace_cats(X):
    for col in cat_map.keys():        
        if col in X.columns:
            X[col] = X[col].replace(cat_map[col])
    return X

# ct = ColumnTransformer(transformers=[
#     ('full_date', FunctionTransformer(), ['MoSold', 'YrSold'])
# ], remainder='passthrough', verbose_feature_names_out=False)

# X_trans = ct.fit_transform(X)
# X_trans['Foundation'].value_counts(normalize=True)

In [134]:
class BsmtDetailsReplacer(BaseEstimator, TransformerMixin):
    def __init__(self, BsmtCond):
        self.BsmtCond = BsmtCond
        self.y = None
    def fit(self, X, y=None):        
        return self
    def transform(self, X):
        for col in X.columns:
            X.loc[(self.BsmtCond.notna()) & (X[col].isna()), col] = X[col].mode()[0]
        return X

BsmtExposure_pl = Pipeline(steps=[
    ('impute', BsmtDetailsReplacer(X['BsmtCond'])),
    ('fill_na', SimpleImputer(strategy='constant', fill_value='NB'))
])
BsmtDetails_pl = Pipeline(steps=[
    ('impute', BsmtDetailsReplacer(X['BsmtCond'])),
    ('fill_na', SimpleImputer(strategy='constant', fill_value='No'))
])

In [135]:
drop_cols = ['Alley', 'Condition2', 'Exterior2nd', 'MoSold', 'YrSold', 'BsmtFinType2', 'MiscFeature', 'BsmtFinSF1', 'BsmtFinSF2', 'Utilities', 'LowQualFinSF', '1stFlrSF']
Bsmt_details_cols = ['BsmtFinType1']
impute_zero_cols = ['BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageYrBlt', 'MasVnrArea']
impute_No_cols = ['BsmtQual', 'BsmtCond', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence']
impute_mean_cols = ['LotFrontage', 'GarageCars', 'GarageArea']
impute_mode_cols = ['Electrical', 'KitchenQual', 'Functional']

preprocessing_ct = ColumnTransformer(transformers=[
    ('BsmtExposure_prep', BsmtExposure_pl, ['BsmtExposure']),
    ('BsmtDetails_prep', BsmtDetails_pl, Bsmt_details_cols),
    ('impute_zero', SimpleImputer(strategy='constant', fill_value=0), impute_zero_cols),
    ('impute_No', SimpleImputer(strategy='constant', fill_value='No'), impute_No_cols),
    ('impute_mean', SimpleImputer(strategy='mean'), impute_mean_cols),
    ('impute_mode', SimpleImputer(strategy='most_frequent'), impute_mode_cols),
    ('replace_cats', FunctionTransformer(replace_cats), replace_cats_cols),
    ('drop', 'drop', drop_cols)
], remainder='passthrough', verbose_feature_names_out=False)


In [136]:
def add_cols_naming(transformer_name, feature_name):
    if (transformer_name == 'keep') | (transformer_name == 'remainder'):
        name = feature_name
    else:
        name = transformer_name
    return name

HouseStyles2ndFloor = (['2Story', '2.5Fin', '2.5Unf', '1.5Fin'])

add_ct = ColumnTransformer(transformers=[
    ('HasGarage', FunctionTransformer(lambda x: x != 0), ['GarageYrBlt']),
    ('HasBsmt', FunctionTransformer(lambda x: x != 'No'), ['BsmtQual']),
    ('HasPool', FunctionTransformer(lambda x: x != 0), ['PoolArea']),
    ('Has2ndFloor', FunctionTransformer(lambda x: pd.DataFrame((x['HouseStyle'].isin(HouseStyles2ndFloor)) * (x['2ndFlrSF'] > 100))), ['HouseStyle', '2ndFlrSF']),
    ('BsmtUnfPerc', FunctionTransformer(lambda x: pd.DataFrame(x['BsmtUnfSF'] / x['TotalBsmtSF'])), ['BsmtUnfSF', 'TotalBsmtSF']),
    ('TotalLivArea', FunctionTransformer(lambda x: pd.DataFrame(x['GrLivArea'] + x['TotalBsmtSF'])), ['TotalBsmtSF', 'GrLivArea']),
    ('keep', 'passthrough', ['GarageYrBlt', 'BsmtQual', 'PoolArea', 'TotalBsmtSF', 'HouseStyle'])
], remainder='passthrough', verbose_feature_names_out=add_cols_naming)

# X_trans = add_ct.fit_transform(X)
# sns.kdeplot(data=X_trans.join(y), x='DateSold', y='SalePrice')

In [137]:
Ordinal_cols = ['LotShape', 'LandSlope', 'ExterQual', 'ExterCond',
                'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
                'HeatingQC', 'KitchenQual', 'Functional', 'FireplaceQu',
                'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive',
                'PoolQC', 'Fence']
LotShape_cats = ['Reg','IR1','IR2','IR3']
LandSlope_cats = ['Gtl', 'Mod', 'Sev']
ExterQual_cats = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
BsmtQual_cats = ['Ex', 'Gd', 'TA', 'Fa', 'Po', 'No']
BsmtExposure_cats = ['Gd', 'Av', 'Mn', 'No', 'NB']
BsmtFinType1_cats = ['No', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
Functional_cats = ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ']
GarageFinish_cats = ['No', 'Unf', 'RFn', 'Fin']
PavedDrive_cats = ['Y', 'P', 'N']
PoolQC_cats = ['No', 'Fa', 'TA', 'Gd', 'Ex']
Fence_cats = ['No', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']

Order_encoder = OrdinalEncoder(categories=[LotShape_cats, LandSlope_cats, ExterQual_cats, ExterQual_cats,
                                           BsmtQual_cats, BsmtQual_cats, BsmtExposure_cats, BsmtFinType1_cats,
                                           ExterQual_cats, ExterQual_cats, Functional_cats, BsmtQual_cats,
                                           GarageFinish_cats, BsmtQual_cats, BsmtQual_cats, PavedDrive_cats,
                                           PoolQC_cats, Fence_cats],
                                           encoded_missing_value=-1)

In [138]:
OH_cols = ['MSZoning', 'Street', 'LandContour', 'LotConfig', 'Condition1', 'RoofStyle',
           'BldgType', 'HouseStyle', 'RoofMatl', 'Exterior1st', 'MasVnrType', 'Foundation', 'Heating',
           'CentralAir', 'Electrical', 'GarageType', 'SaleType', 'SaleCondition']
OH_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

In [139]:
encode_ct = ColumnTransformer(transformers=[
    ('ordered', Order_encoder, Ordinal_cols),
    ('onehot', OH_encoder, OH_cols)
], remainder='passthrough')

In [140]:
forest_model = RandomForestRegressor(n_estimators=375, max_depth=33, max_features=0.25, random_state=0)
# xgb_model = xgb.XGBRegressor(n_estimators=250, max_depth=20, learning_rate=0.05, n_jobs=2, early_stopping_rounds=5)

# preproc_pl = Pipeline(steps=[
#     ('preproc', preprocessing_ct),
#     ('add_cols', add_ct),
#     ('encode', encode_ct)
# ])
# X_trans = preproc_pl.fit_transform(X, y)
# HTML(pd.DataFrame(X_trans.corrwith(y).abs().sort_values(ascending=False)).to_html())

model_pl = Pipeline(steps=[
    ('preproc', preprocessing_ct),
    ('add_cols', add_ct),
    ('encode', encode_ct),
    ('model', forest_model)
])

# X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

# preproc_pl.fit(X_train, y_train)
# X_val_xgb = preproc_pl.transform(X_val)
# model_pl.fit(X_train, y_train, model__eval_set=[(X_val_xgb, y_val)], model__verbose=False)

model_pl.fit(X, y)

# print("MAE (Your approach):")
# print((-1 * cross_val_score(model_pl, X, y, scoring='neg_mean_absolute_error')).mean())

# def score_per_param(param):
#     param_pl = Pipeline(steps=[
#         ('preproc', preprocessing_ct),
#         ('add_cols', add_ct),
#         ('encode', encode_ct),
#         ('model', RandomForestRegressor(n_estimators=375, max_depth=33, max_features=param, random_state=0))
#     ])
#     return (-1 * cross_val_score(param_pl, X, y, scoring='neg_mean_absolute_error')).mean()
# {n : score_per_param(n) for n in np.arange(0.2, 0.35, 0.05)}

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preproc', ...), ('add_cols', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('BsmtExposure_prep', ...), ('BsmtDetails_prep', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If t

In [141]:
X_test = pd.read_csv('test.csv', index_col='Id')
X_test[m_estimate_cols] = m_encoder.transform(X_test[m_estimate_cols])

pred_test = model_pl.predict(X_test)

In [142]:
output = pd.DataFrame({'Id': X_test.index, 'SalePrice': pred_test})
output.to_csv('submission.csv', index=False)